# 🤟 한국수어 제스처 인식 (Korean Sign Language Recognition)
**데이터셋**: LeapGestRecog (Kaggle) — 10개 제스처 클래스 × 10명 피험자  
**방법론**: MediaPipe Hands 랜드마크 추출 → 머신러닝 분류 (RandomForest / SVM / MLP)  
**목표**: 실시간 수어 단어 인식 서비스 프로토타입 구축  
**향후 확장**: AI Hub 한국수어 데이터셋 적용 (30개 단어)

In [ ]:
# ── 0. GPU 확인 및 패키지 설치 ─────────────────────────────
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
print('GPU 이름:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

!pip install -q gradio mediapipe opencv-python-headless

In [ ]:
# ── 1. 라이브러리 임포트 및 전역 상수 ──────────────────────
import os, random, warnings
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib
warnings.filterwarnings('ignore')

# 전역 상수
SEED        = 42
BASE_PATH   = '/content/gesture/leapGestRecog'
NUM_CLASSES = 10
TEST_SIZE   = 0.2
VAL_SIZE    = 0.1

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print('시드 고정 완료:', SEED)

In [ ]:
# ── 2. 한글 폰트 설정 ───────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
nanum = [f for f in fm.findSystemFonts() if 'Nanum' in f]
if nanum:
    plt.rcParams['font.family'] = fm.FontProperties(fname=nanum[0]).get_name()
    plt.rcParams['axes.unicode_minus'] = False
    print('한글 폰트 설정 완료:', nanum[0])

# 폰트 테스트
fig, ax = plt.subplots(figsize=(5, 1))
ax.text(0.5, 0.5, '한글 폰트 테스트: 한국수어 인식', ha='center', va='center', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Kaggle API 인증 및 데이터셋 다운로드 ────────────────
from google.colab import files
print('kaggle.json 파일을 업로드하세요')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!mkdir -p /content/gesture
!kaggle datasets download -d gti-upm/leapgestrecog -p /content/gesture --unzip -q
print('다운로드 완료!')
print('폴더 구조:')
!ls /content/gesture/leapGestRecog | head -5

In [ ]:
# ── 4. 데이터셋 구조 탐색 ───────────────────────────────────
# LeapGestRecog → 10명 피험자(00~09) × 10개 클래스
base = Path(BASE_PATH)
subjects = sorted([d.name for d in base.iterdir() if d.is_dir()])
sample_subject = base / subjects[0]
classes_raw = sorted([d.name for d in sample_subject.iterdir() if d.is_dir()])

print(f'피험자 수: {len(subjects)}명  → {subjects}')
print(f'제스처 클래스: {len(classes_raw)}개')
for c in classes_raw:
    n = len(list((sample_subject / c).glob('*.png')))
    print(f'  {c}: {n}개 이미지 (피험자 1명 기준)')

# 10개 클래스 → 한국수어 단어 매핑
GESTURE_KO = {
    '01_palm'       : '안녕하세요',
    '02_l'          : '좋아요',
    '03_fist'       : '화나요',
    '04_fist_moved' : '먹다',
    '05_thumb'      : '괜찮아요',
    '06_index'      : '네',
    '07_ok'         : '감사합니다',
    '08_palm_moved' : '가다',
    '09_c'          : '아파요',
    '10_down'       : '아니요',
}
print('\n제스처 → 한국수어 매핑:')
for k, v in GESTURE_KO.items():
    print(f'  {k:20s} → {v}')

In [ ]:
# ── 5. 샘플 이미지 시각화 ────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('LeapGestRecog 제스처 클래스 샘플 (피험자 00)', fontsize=15, fontweight='bold')

for ax, cls in zip(axes.flatten(), classes_raw):
    imgs = list((base / subjects[0] / cls).glob('*.png'))
    if imgs:
        img = cv2.imread(str(imgs[0]), cv2.IMREAD_GRAYSCALE)
        ax.imshow(img, cmap='gray')
        ko_label = GESTURE_KO.get(cls, cls)
        ax.set_title(f'{cls}\n→ {ko_label}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── 6. 클래스별 이미지 수 분포 시각화 ───────────────────────
count_data = {}
for cls in classes_raw:
    total = sum(len(list((base / subj / cls).glob('*.png'))) for subj in subjects)
    count_data[GESTURE_KO[cls]] = total

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(count_data.keys(), count_data.values(),
              color=plt.cm.Set2(np.linspace(0, 1, 10)), edgecolor='black', linewidth=0.7)
ax.set_title('클래스별 전체 이미지 수 분포', fontsize=13, fontweight='bold')
ax.set_xlabel('제스처 (한국수어 단어)', fontsize=11)
ax.set_ylabel('이미지 수', fontsize=11)
for bar, v in zip(bars, count_data.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(v), ha='center', fontsize=9)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
print(f'총 이미지 수: {sum(count_data.values()):,}개')

In [ ]:
# ── 7. MediaPipe 손 랜드마크 추출 함수 ─────────────────────
mp_hands = mp.solutions.hands

def extract_landmarks(img_path: str) -> list | None:
    """
    이미지에서 MediaPipe 21개 손 랜드마크를 추출해 42차원 벡터로 반환.
    손이 감지되지 않으면 None 반환.
    """
    img = cv2.imread(img_path)
    if img is None:
        return None
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:
        result = hands.process(rgb)

    if not result.multi_hand_landmarks:
        return None

    hl = result.multi_hand_landmarks[0]
    pts = np.array([[lm.x, lm.y] for lm in hl.landmark], dtype=np.float32)

    # 정규화: 손목(0번) 기준 평행이동 + landmark 9 거리로 스케일
    pts -= pts[0]
    scale = float(np.linalg.norm(pts[9]))
    if scale > 1e-6:
        pts /= scale

    return pts.flatten().tolist()

# 단일 이미지 테스트
test_img = str(list((base / subjects[0] / classes_raw[0]).glob('*.png'))[0])
vec = extract_landmarks(test_img)
print(f'랜드마크 추출 결과: {"성공" if vec else "실패"}')
if vec:
    print(f'특징 벡터 차원: {len(vec)}  (21개 관절 × x,y = 42)')

In [ ]:
# ── 8. 전체 데이터셋 랜드마크 추출 (배치 처리) ─────────────
# ※ 전체 추출 시 수 분 소요 — 진행률 출력
from tqdm.notebook import tqdm

rows = []
skipped = 0

for subj in tqdm(subjects, desc='피험자'):
    for cls in classes_raw:
        img_paths = list((base / subj / cls).glob('*.png'))
        for path in img_paths:
            vec = extract_landmarks(str(path))
            if vec is not None:
                rows.append([cls] + vec)
            else:
                skipped += 1

cols = ['label'] + [f'{ax}{i}' for i in range(21) for ax in ('x', 'y')]
df = pd.DataFrame(rows, columns=cols)

# 한국수어 레이블 추가
df['label_ko'] = df['label'].map(GESTURE_KO)

print(f'\n추출 완료: {len(df):,}개 샘플 | 제외(손 미감지): {skipped}개')
print(f'클래스별 샘플 수:')
print(df['label_ko'].value_counts().to_string())

# CSV 저장
df.to_csv('/content/landmarks_extracted.csv', index=False, encoding='utf-8-sig')
print('\n저장 완료: /content/landmarks_extracted.csv')

In [ ]:
# ── 9. 랜드마크 시각화 — 제스처별 평균 손 모양 ─────────────
mp_draw = mp.solutions.drawing_utils
mp_conn = mp_hands.HAND_CONNECTIONS

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('클래스별 평균 랜드마크 형태', fontsize=14, fontweight='bold')

feature_cols = [f'{ax}{i}' for i in range(21) for ax in ('x', 'y')]

for ax, cls in zip(axes.flatten(), classes_raw):
    subset = df[df['label'] == cls][feature_cols].values
    if len(subset) == 0:
        ax.axis('off')
        continue
    mean_pts = subset.mean(axis=0).reshape(21, 2)
    # y축 반전 (이미지 좌표계)
    mean_pts[:, 1] = -mean_pts[:, 1]
    # 관절 연결선
    connections = [
        (0,1),(1,2),(2,3),(3,4),
        (0,5),(5,6),(6,7),(7,8),
        (0,9),(9,10),(10,11),(11,12),
        (0,13),(13,14),(14,15),(15,16),
        (0,17),(17,18),(18,19),(19,20),
        (5,9),(9,13),(13,17)
    ]
    for s, e in connections:
        ax.plot([mean_pts[s,0], mean_pts[e,0]],
                [mean_pts[s,1], mean_pts[e,1]], 'b-', lw=1.5, alpha=0.6)
    ax.scatter(mean_pts[:,0], mean_pts[:,1], c='red', s=30, zorder=5)
    ax.set_title(f'{GESTURE_KO[cls]}', fontsize=10, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── 10. 학습/검증/테스트 분할 ───────────────────────────────
feature_cols = [f'{ax}{i}' for i in range(21) for ax in ('x', 'y')]
X = df[feature_cols].values
y = df['label'].values

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y_enc, test_size=TEST_SIZE, stratify=y_enc, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=VAL_SIZE/(1-TEST_SIZE),
    stratify=y_trainval, random_state=SEED)

print(f'학습:  {len(X_train):>6,}개')
print(f'검증:  {len(X_val):>6,}개')
print(f'테스트:{len(X_test):>6,}개')
print(f'특징 차원: {X.shape[1]}')
print(f'클래스: {list(le.classes_)}')

In [ ]:
# ── 11. 모델 학습 — RandomForest / SVM / MLP 비교 ──────────
models = {
    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED, probability=True))
    ]),
    'MLP': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(256, 128, 64),
                              activation='relu', max_iter=300,
                              random_state=SEED, early_stopping=True,
                              validation_fraction=0.1))
    ]),
}

results = {}
for name, model in models.items():
    print(f'학습 중: {name} ...')
    model.fit(X_train, y_train)
    val_acc  = accuracy_score(y_val,  model.predict(X_val))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = {'val_acc': val_acc, 'test_acc': test_acc, 'model': model}
    print(f'  검증 정확도: {val_acc:.4f}  |  테스트 정확도: {test_acc:.4f}')

best_name = max(results, key=lambda k: results[k]['test_acc'])
best_model = results[best_name]['model']
print(f'\n최고 모델: {best_name}  (테스트 정확도: {results[best_name]["test_acc"]:.4f})')

In [ ]:
# ── 12. 모델 성능 비교 시각화 ───────────────────────────────
names     = list(results.keys())
val_accs  = [results[n]['val_acc']  * 100 for n in names]
test_accs = [results[n]['test_acc'] * 100 for n in names]

x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - 0.2, val_accs,  0.35, label='검증 정확도', color='#4472C4', alpha=0.85)
b2 = ax.bar(x + 0.2, test_accs, 0.35, label='테스트 정확도', color='#ED7D31', alpha=0.85)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylabel('정확도 (%)', fontsize=11)
ax.set_title('모델별 정확도 비교', fontsize=13, fontweight='bold')
ax.set_ylim(0, 110)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 13. 혼동 행렬 (최고 모델) ───────────────────────────────
y_pred = best_model.predict(X_test)
ko_labels = [GESTURE_KO[c] for c in le.classes_]
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=ko_labels, yticklabels=ko_labels,
            linewidths=0.5, ax=ax)
ax.set_title(f'혼동 행렬 — {best_name}', fontsize=14, fontweight='bold')
ax.set_xlabel('예측 레이블', fontsize=11)
ax.set_ylabel('실제 레이블', fontsize=11)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── 14. 분류 보고서 ─────────────────────────────────────────
print(f'=== {best_name} 분류 보고서 ===')
print(classification_report(y_test, y_pred,
      target_names=ko_labels, digits=4))

# 클래스별 정확도 바 차트
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#37863D' if a >= 90 else '#FFC000' if a >= 70 else '#C00000'
          for a in per_class_acc]
bars = ax.bar(ko_labels, per_class_acc, color=colors, edgecolor='black', linewidth=0.6)
for bar, v in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v:.1f}%', ha='center', fontsize=9)
ax.set_title('클래스별 인식 정확도', fontsize=13, fontweight='bold')
ax.set_ylabel('정확도 (%)', fontsize=11)
ax.set_ylim(0, 115)
ax.axhline(90, color='green',  ls='--', alpha=0.5, label='90% 기준선')
ax.axhline(70, color='orange', ls='--', alpha=0.5, label='70% 기준선')
ax.legend(fontsize=9)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 15. 모델 저장 ────────────────────────────────────────────
import json

joblib.dump(best_model, '/content/sign_model.pkl')
joblib.dump(le, '/content/label_encoder.pkl')

meta = {
    'model_name'   : best_name,
    'test_accuracy': round(results[best_name]['test_acc'], 4),
    'num_classes'  : NUM_CLASSES,
    'feature_dim'  : 42,
    'classes_en'   : list(le.classes_),
    'classes_ko'   : ko_labels,
    'gesture_map'  : GESTURE_KO,
    'dataset'      : 'LeapGestRecog (Kaggle)'
}
with open('/content/model_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('저장 완료:')
print('  /content/sign_model.pkl')
print('  /content/label_encoder.pkl')
print('  /content/model_meta.json')
print(f'\n모델: {best_name}  |  테스트 정확도: {results[best_name]["test_acc"]:.4f}')

In [ ]:
# ── 16. Gradio 실시간 추론 인터페이스 ───────────────────────
import gradio as gr
from PIL import Image

model_inf = joblib.load('/content/sign_model.pkl')
le_inf    = joblib.load('/content/label_encoder.pkl')

with open('/content/model_meta.json', encoding='utf-8') as f:
    meta_inf = json.load(f)

def predict_gesture(pil_img):
    img_np = np.array(pil_img.convert('RGB'))
    img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    with mp_hands.Hands(static_image_mode=True, max_num_hands=1,
                        min_detection_confidence=0.5) as hands:
        result = hands.process(img_rgb)

    if not result.multi_hand_landmarks:
        return '손이 감지되지 않았습니다. 손을 화면에 정면으로 보여주세요.'

    hl  = result.multi_hand_landmarks[0]
    pts = np.array([[lm.x, lm.y] for lm in hl.landmark], dtype=np.float32)
    pts -= pts[0]
    scale = float(np.linalg.norm(pts[9]))
    if scale > 1e-6:
        pts /= scale
    vec = pts.flatten().reshape(1, -1)

    pred_enc  = model_inf.predict(vec)[0]
    pred_prob = model_inf.predict_proba(vec)[0]
    pred_cls  = le_inf.inverse_transform([pred_enc])[0]
    pred_ko   = meta_inf['gesture_map'].get(pred_cls, pred_cls)
    confidence = pred_prob.max() * 100

    top3_idx = pred_prob.argsort()[-3:][::-1]
    top3 = [(meta_inf['gesture_map'].get(le_inf.classes_[i], le_inf.classes_[i]),
             pred_prob[i]*100) for i in top3_idx]

    result_str = f'🤟 인식 결과: {pred_ko}\n'
    result_str += f'신뢰도: {confidence:.1f}%\n\n'
    result_str += 'Top-3 예측:\n'
    for i, (ko, prob) in enumerate(top3):
        result_str += f'  {i+1}. {ko}: {prob:.1f}%\n'
    return result_str

demo = gr.Interface(
    fn=predict_gesture,
    inputs=gr.Image(type='pil', label='수어 이미지 업로드'),
    outputs=gr.Textbox(label='인식 결과', lines=8),
    title='🤟 한국수어 제스처 인식',
    description=(
        f'LeapGestRecog 데이터셋 기반 수어 인식 데모\n'
        f'인식 가능 단어: {", ".join(ko_labels)}\n'
        f'모델: {best_name}  |  테스트 정확도: {results[best_name]["test_acc"]*100:.1f}%'
    ),
    examples=[],
    flagging_mode='never'
)

demo.launch(share=True, debug=False)

## 📋 향후 확장 계획

| 단계 | 내용 | 일정 |
|------|------|------|
| **현재** | LeapGestRecog 10개 클래스 → 한국수어 매핑 프로토타입 | 완료 |
| **2단계** | AI Hub 한국수어 영상 데이터셋 다운로드 및 랜드마크 추출 | ~2026.06.06 |
| **3단계** | 30개 한국수어 단어 전체 학습 및 모델 고도화 | 2026.06.07~ |
| **4단계** | 실시간 웹캠 추론 + gTTS 음성 출력 통합 | 미정 |

### 현재 인식 가능 단어 (10개)
안녕하세요, 좋아요, 화나요, 먹다, 괜찮아요, 네, 감사합니다, 가다, 아파요, 아니요

### AI Hub 적용 후 추가 예정 단어 (20개)
감사합니다, 죄송합니다, 잠깐만요, 싫어요, 기뻐요, 슬퍼요, 모르겠어요, 도와주세요,
밥, 물, 화장실, 집, 병원, 나, 너, 엄마, 아빠, 친구, 마시다, 오다, 자다